# FSF fit inspection
This notebook loads in a fastspecfit (FSF) model with a given targetid and 
plots the observed and model spectra. 

The goal is to understand why or how any systemic residuals arise and if FSF is reliable. 

The data files are to be downloaded via `dowload_spectra.py`

In [ ]:
from astropy.table import Table

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import arya
arya.use("mnras")

In [ ]:
import tomllib

In [ ]:
import fitsio

In [ ]:
from desiutil.dust import dust_transmission

In [ ]:
from desispec.io import read_spectra
from desispec.coaddition import coadd_cameras

In [ ]:
from pathlib import Path

In [ ]:
import sys
sys.path.append("..")

In [ ]:
from desi_utils.catalogue_helpers import FSF_LINES, get_fsf_wave

In [ ]:
from desi_utils.get_data import get_spectra
from desi_utils import Spectrum

In [ ]:
from desi_utils.plotting import plot_wrapped_spectrum

In [ ]:
from astropy import constants as c

In [ ]:
C_KMS = c.c.to("km/s").value

# Data loading

In [ ]:
with open("../analysis/selected_targetid.txt", "rb") as f:
    targetids = tomllib.load(f)["targetid"]

In [ ]:
def read_fsf_file(file):
    meta = Table(fitsio.read(file, "METADATA"))
    fast = Table(fitsio.read(file, "FASTSPEC"))
    models, hdr = fitsio.read(file, "MODELS", header=True)
    modelwave = hdr["CRVAL1"] + np.arange(hdr["NAXIS1"]) * hdr["CDELT1"]

    return meta, fast, modelwave, models

In [ ]:
targetid = targetids[13]

In [ ]:
out_dir = Path(".", "example_fits", str(targetid))

In [ ]:
if not out_dir.exists():
    out_dir.mkdir()

In [ ]:
desi_file = out_dir / "coadd.fits"

In [ ]:
fsf_file = out_dir / "fsf.fits"

In [ ]:
meta, fast, modelwave, model_spectra = read_fsf_file(fsf_file)

In [ ]:
E_BV =  meta["EBV"]

# Plotting

In [ ]:
def get_spectrum(meta, spec_id):
    desi_spec = read_spectra(desi_file).select(targets = meta["TARGETID"])
    E_BV =  meta["EBV"]
    desi_coadd = coadd_cameras(desi_spec)

    spec_wave = desi_coadd.wave[desi_coadd.bands[0]]
    spec_flux = desi_coadd.flux[desi_coadd.bands[0]][spec_id, :]
    spec_ivar = desi_coadd.ivar[desi_coadd.bands[0]][spec_id, :]
    spec_mask = desi_coadd.mask[desi_coadd.bands[0]][spec_id, :]
    mw_trans = dust_transmission(spec_wave, E_BV[spec_id])
    spec_flux[spec_mask] = np.nan
    spec_undust = spec_flux / mw_trans

    
    spec = Spectrum(
        wavelength = spec_wave,
        flux = spec_undust,
        uncertainty = spec_ivar ** -0.5  / mw_trans,
        redshift = meta["Z"][spec_id],
    )

    return spec

In [ ]:
def get_raw_spectra(meta):
    return read_spectra(desi_file).select(targets = meta["TARGETID"])


In [ ]:
spec_raw = get_raw_spectra(meta)

In [ ]:
spec_coadd = coadd_cameras(spec_raw)

In [ ]:
import desispec

In [ ]:
R = desispec.resolution.Resolution(spec_coadd.resolution_data["brz"][spec_id, :, :])

In [ ]:
spec_id = np.where(meta["TARGETID"] == targetid)[0][0]

In [ ]:
model_stars = model_spectra[spec_id, 0, :] + model_spectra[spec_id, 1, :]
model_gas = model_spectra[spec_id, 2, :]
model_tot = model_gas + model_stars

In [ ]:
spec = get_spectrum(meta, spec_id)

In [ ]:
spec_dict = {
    "observed": spec,
    "model": (modelwave, model_tot)
}

In [ ]:
styles = {
    "observed": dict(color="k", lw=2),
    "model": dict(color="C1", lw=1.5)
}

In [ ]:
plot_wrapped_spectrum(spec_dict, styles, show_uncertainty=True) 
plt.savefig(out_dir / "fsf_model_spectra.pdf")

In [ ]:
plot_wrapped_spectrum(spec_dict, styles, yscale="log", show_uncertainty=True) 
plt.savefig(out_dir / "fsf_model_log_spectra.pdf")

In [ ]:
linefit_dir = out_dir / "fsf_lines"
if not linefit_dir.exists():
    linefit_dir.mkdir()

In [ ]:
from desi_utils.fit_continuum import redshift_to_vel, vel_to_redshift


In [ ]:
from desi_utils.fit_lines import _gaussian_flux

In [ ]:
import desi_utils

In [ ]:
from scipy.special import erf

In [ ]:
np.array(R)

In [ ]:
np.mean(R.transpose() == R)

In [ ]:
def plot_gaussian(x_model, mu, sigma, flux, cont, R=R, **kwargs):
    amp = flux / sigma / np.sqrt(2*np.pi)
    lam0 = mu 
    
    x_edges = tuple(e[:, None] for e in desi_utils.fit_lines._pixel_edges(x_model))

    y_model = _gaussian_flux(
        x_model, amp, 
        lam0, sigma,
        edges = x_edges, 
        exp = np.exp, 
        erf = erf
    )[:, 0]

    # plt.plot(x_model, y_model, **kwargs)
    y_model_new = R * y_model

    y_model_new += cont
    plt.plot(x_model, y_model_new, **kwargs)


In [ ]:
EM_WAVELENGTHS = [desi_utils.get_wavelength(line) for line in desi_utils.EMISSION_LINES]

In [ ]:
def get_precise_wave(fsf_line):
    wave0 = get_fsf_wave(fsf_line)
    idx = np.argmin(np.abs(wave0 - np.array(EM_WAVELENGTHS)))
    wave = EM_WAVELENGTHS[idx]
    assert abs(wave / wave0 - 1) < 0.01
    return wave

In [ ]:
def get_fsf_model(fast, fsf_line, spec_id):
    v = fast[fsf_line + "_VSHIFT"][spec_id] 
    lambda0 = get_precise_wave(fsf_line) * (1 + meta["Z"][spec_id] + v / C_KMS)

    sigma = fast[fsf_line + "_SIGMA"][spec_id] / C_KMS * lambda0

    mu = lambda0

    flux = fast[fsf_line + "_FLUX"][spec_id]
    return mu, sigma, flux

In [ ]:
def plot_line_region(line):
    x0 = get_fsf_wave(line) * (1 + spec.redshift)

    dx = 30
    filt = spec.wavelength > x0 - dx
    filt &= spec.wavelength < x0 + dx

    xlim = x0-dx, x0+dx
    fig, axs = plt.subplots(2, sharex=True)
    
    plt.sca(axs[0])
    chi2 = fast[line + "_CHI2"][spec_id]
    plt.plot(spec.wavelength[filt], spec.flux[filt], color="k")
    plt.fill_between(spec.wavelength[filt], spec.flux[filt] - spec.uncertainty[filt], spec.flux[filt] + spec.uncertainty[filt], color="k", alpha=0.2)
    plt.plot(spec.wavelength[filt], model_tot[filt], color="C1", zorder=5, lw=1)

    # plot the gaussian
    mu, sigma, flux = get_fsf_model(fast, line, spec_id)
    plot_gaussian(spec.wavelength, mu, sigma, flux, model_stars)
    plt.xlim(xlim)

    
    # plt.plot(spec.wavelength[filt], spec.meta["model"][filt])
    plt.title(line.replace("_", " ") + f", $\\chi^2$ = {chi2:0.3g}")
    plt.xlabel("")
    plt.ylabel("flux")

    plt.sca(axs[1])
    plt.plot(spec.wavelength[filt], spec.flux[filt] - model_tot[filt], color="k")
    plt.fill_between(spec.wavelength[filt], spec.flux[filt] - spec.uncertainty[filt] - model_tot[filt], spec.flux[filt] + spec.uncertainty[filt] - model_tot[filt], color="k", alpha=0.2)

    plt.xlim(xlim)
    plt.axhline(0, color="C1", lw=1)
    plt.xlabel("$\\lambda$ observed / $\\AA$")
    plt.ylabel("residual")
    plt.savefig(linefit_dir / (line + ".pdf"))
    plt.show()

In [ ]:
for line in FSF_LINES:
    x0 = get_fsf_wave(line) * (1 + spec.redshift)


    if line in ["OII_3729", "H6_BROAD", "HEPSILON_BROAD", "HGAMMA_BROAD", "HBETA_BROAD", "HALPHA_BROAD"]:
        continue

    if x0 > spec.wavelength[0] and x0 < spec.wavelength[-1]:
        plot_line_region(line)


# SPARCL

In [ ]:
spec_sparcl = desi_utils.get_data.get_spectra(targetid)

In [ ]:
plot_wrapped_spectrum(
    {"SPARCL": Spectrum.from_desi(spec_sparcl),
     "desi": spec
    },
    {
        "desi": dict(lw=0.5)
    }
)

In [ ]:
mw_trans = dust_transmission(spec.wavelength, E_BV[spec_id])


In [ ]:
plot_wrapped_spectrum(
    {"SPARCL": Spectrum.from_desi(spec_sparcl) * (1/mw_trans),
     "desi": spec
    },
    {
        "desi": dict(lw=0.5)
    }
)

In [ ]:
plot_wrapped_spectrum(
    {"SPARCL": (spec.wavelength, Spectrum.from_desi(spec_sparcl).uncertainty * (1/mw_trans)),
     "desi": (spec.wavelength, spec.uncertainty)
    },
    {
        "desi": dict(lw=0.5)
    }
)

In [ ]:
plot_wrapped_spectrum(
    {
         "desi": spec * mw_trans,
        "b": (spec_raw.wave["b"], spec_raw.flux["b"][spec_id, :]),
        "r": (spec_raw.wave["r"], spec_raw.flux["r"][spec_id, :]),
        "z": (spec_raw.wave["z"], spec_raw.flux["z"][spec_id, :]),
    },
    {
        "b": dict(lw=0.5, color="C0"),
        "r": dict(lw=0.5, color="C3"),
        "z": dict(lw=0.5, color="C1"),
        "desi": dict(lw=2, color='k'),
    }
)